# Aula 22 — Processamento Batch vs. Streaming

**Disciplina:** Big Data e Inteligência Artificial  
**Data:** 17/11/2026  
**Fonte:** `samples.wanderbricks.clickstream` — eventos de navegação disponibilizados pelo Databricks  
**Ambiente:** Databricks Free Edition (compute serverless)

## Objetivo

Comparar processamento **batch** e **streaming** utilizando os **mesmos eventos**, com quatro ciclos progressivos: leitura inicial, chegada de dados, ingestão incremental e retomada da consulta.

**Importante:** os eventos são registros de uma tabela histórica do catálogo `samples`. Nós apenas **reproduzimos sua chegada** em uma tabela Delta própria; não existe uma fonte online de eventos nesta aula. Na aula seguinte, o caso será MQTT real.

**Roteiro:** execute as células em ordem. As atividades `TODO` são resolvidas pelos alunos na versão de estudante.

**Versão do professor:** atividades resolvidas e orientações de condução.

## Preparação — catálogo `samples`, tabelas de trabalho e checkpoint

O catálogo `samples` é **somente leitura**: não o modificaremos. Copiaremos **30 eventos existentes** para uma amostra de trabalho, com um número sequencial `id_evento` acrescentado apenas para organizar os três lotes didáticos.

As tabelas de trabalho serão criadas no **seu catálogo e esquema**, e o checkpoint ficará em um **Volume do Unity Catalog**.

**Antes de executar:** confira o `catalogo` e o `esquema` na célula. Se o catálogo atual não permitir criação de tabelas/Volumes, substitua pelos nomes em que sua conta tem permissão.

**Reinício:** a célula de preparação recria apenas os objetos identificados como `aula22_*`. Para refazer a experiência, execute esta preparação e todos os ciclos na ordem. **Não reexecute apenas uma célula que adiciona um lote**, pois isso poderá inserir eventos duplicados na fonte.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Altere somente se seu workspace utilizar outro catálogo ou esquema.
catalogo = spark.sql("SELECT current_catalog()").first()[0]
esquema = "default"

origem_samples = "samples.wanderbricks.clickstream"
amostra = f"`{catalogo}`.`{esquema}`.`aula22_amostra`"
fonte = f"`{catalogo}`.`{esquema}`.`aula22_fonte`"
destino = f"`{catalogo}`.`{esquema}`.`aula22_destino`"
volume = "aula22_streaming"

# Checkpoint exclusivo da atividade.
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalogo}`.`{esquema}`.`{volume}`")
checkpoint = f"/Volumes/{catalogo}/{esquema}/{volume}/checkpoint_clickstream"

# Reinicia somente os objetos da aula.
dbutils.fs.rm(checkpoint, True)
spark.sql(f"DROP TABLE IF EXISTS {destino}")
spark.sql(f"DROP TABLE IF EXISTS {fonte}")
spark.sql(f"DROP TABLE IF EXISTS {amostra}")

# Seleciona registros da tabela pública. O id_evento é criado para dividir os lotes.
base = (spark.table(origem_samples)
    .select(F.col("event").alias("evento"),
            F.col("metadata.device").alias("dispositivo"))
    .where(F.col("evento").isNotNull())
    .limit(120)
    .withColumn("id_evento", F.row_number().over(Window.orderBy(F.rand(seed=22))))
    .filter(F.col("id_evento") <= 30)
    .select("id_evento", "evento", "dispositivo"))

base.write.format("delta").mode("overwrite").saveAsTable(amostra)

# Tabela Delta vazia: receberá os eventos à medida que liberarmos cada lote.
spark.table(amostra).limit(0).write.format("delta").mode("overwrite").saveAsTable(fonte)

print("Fonte pública:", origem_samples)
print("Tabela de entrada:", fonte)
print("Tabela de saída:", destino)
print("Eventos selecionados:", spark.table(amostra).count())
display(spark.table(amostra).orderBy("id_evento"))

**Orientação ao professor:** a amostra é escolhida a partir de uma tabela histórica fornecida pelo Databricks; a reprodução dos eventos é **controlada**, não uma conexão de streaming à internet. O conteúdo de `evento`/`dispositivo` pode variar, mas o exercício deve apresentar **12 → 20 → 30** registros, desde que a tabela pública esteja acessível e tenha ao menos 30 eventos.

No Databricks Free Edition, o `availableNow=True` é compatível com notebook serverless. O mesmo checkpoint é reutilizado entre as execuções da consulta.

## Ciclo 1 — Leitura batch

**Conceito:** processar os registros disponíveis em uma execução delimitada.

Vamos liberar os **12 primeiros eventos** na tabela de entrada e fazer uma leitura batch. O resultado calculado nesta etapa não muda sozinho quando novos registros chegarem.

In [ ]:
# Primeira chegada: 12 eventos do dataset público.
lote1 = spark.table(amostra).filter(F.col("id_evento").between(1, 12))
lote1.write.format("delta").mode("append").saveAsTable(fonte)

# Processamento batch.
df_batch_inicial = spark.read.table(fonte)
total_inicial = df_batch_inicial.count()
print("Quantidade de eventos na primeira análise batch:", total_inicial)
display(df_batch_inicial.orderBy("id_evento"))

**Atividade 1 — Eventos por tipo**

Usando `df_batch_inicial`, calcule a quantidade de eventos para cada valor de `evento`, em ordem decrescente de frequência. Qual evento aparece mais vezes neste primeiro conjunto?

In [ ]:
# Atividade 1 — solução
(df_batch_inicial.groupBy("evento")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False))

**Resultado esperado:** `total_inicial = 12`. A distribuição por tipo depende dos registros presentes na amostra, por isso não há uma contagem fixa por categoria.

## Ciclo 2 — Novos eventos e nova leitura batch

**Conceito:** uma operação batch também pode ser executada novamente. A diferença é que **o resultado já calculado não se atualiza por si só**.

Vamos acrescentar mais **8 eventos** à mesma tabela de entrada.

In [ ]:
# Segunda chegada: mais 8 eventos (IDs 13 a 20).
lote2 = spark.table(amostra).filter(F.col("id_evento").between(13, 20))
lote2.write.format("delta").mode("append").saveAsTable(fonte)

print("Resultado da primeira análise batch (já calculado):", total_inicial)

df_batch_atualizado = spark.read.table(fonte)
print("Nova leitura batch:", df_batch_atualizado.count())
display(df_batch_atualizado.orderBy("id_evento"))

**Atividade 2 — Atualização dos indicadores**

Na nova leitura, conte os eventos por `dispositivo`. Compare o total atual com `total_inicial` e explique por que os dois valores diferem.

In [ ]:
# Atividade 2 — solução
(df_batch_atualizado.groupBy("dispositivo")
    .count()
    .orderBy("dispositivo")
    .show(truncate=False))

print("Total inicial:", total_inicial)
print("Total atualizado:", df_batch_atualizado.count())

**Resultado esperado:** o total inicial continua **12**, enquanto a nova consulta retorna **20**. Streaming não é a única forma de receber dados novos; a diferença está no mecanismo de execução e no acompanhamento do progresso.

## Ciclo 3 — Structured Streaming em tabela Delta

**Conceito:** `readStream` acompanha uma origem que admite novos registros. O checkpoint registra o progresso da consulta. A primeira execução processa os **20 eventos já disponíveis**.

No compute serverless, utilizaremos `.trigger(availableNow=True)`: a consulta processa os dados disponíveis **e encerra**. Isso é streaming **incremental sob demanda**, não uma execução permanentemente ativa.

In [ ]:
# Lê incrementalmente a tabela Delta de entrada.
df_stream = spark.readStream.table(fonte)

consulta = (df_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .toTable(destino))

consulta.awaitTermination()

print("Eventos persistidos pelo streaming:", spark.table(destino).count())
display(spark.table(destino).orderBy("id_evento"))

**Atividade 3 — Consultando a saída do streaming**

Consulte a tabela `destino` com `spark.table()` e conte os eventos por `evento`. Explique: a tabela `samples` está sendo lida continuamente, ou a consulta acompanha a **tabela Delta de entrada** que estamos alimentando?

In [ ]:
# Atividade 3 — solução
(spark.table(destino).groupBy("evento")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False))

print("Fonte monitorada pelo streaming:", fonte)

**Resultado esperado:** **20 registros** no destino. A consulta acompanha a **tabela Delta de entrada**; `samples.wanderbricks.clickstream` foi utilizada somente para obter a amostra histórica.

## Ciclo 4 — Retomada com o mesmo checkpoint

**Conceito:** a próxima execução do streaming deve consumir somente os eventos acrescentados desde o último processamento.

Vamos liberar os **10 eventos restantes** e executar a mesma consulta, com o **mesmo caminho de checkpoint** e a mesma tabela de destino.

In [ ]:
# Terceira chegada: mais 10 eventos (IDs 21 a 30).
lote3 = spark.table(amostra).filter(F.col("id_evento").between(21, 30))
lote3.write.format("delta").mode("append").saveAsTable(fonte)

# A consulta retoma do checkpoint anterior.
consulta = (spark.readStream.table(fonte)
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .toTable(destino))

consulta.awaitTermination()

print("Total da fonte:", spark.table(fonte).count())
print("Total do destino:", spark.table(destino).count())

**Atividade 4 — Verificando a ingestão incremental**

Compare as quantidades de registros na fonte e no destino. Calcule também quantos `id_evento` distintos existem no destino. Por que a segunda execução não deve produzir **50 registros** (20 antigos + 30 totais)?

In [ ]:
# Atividade 4 — solução
print("Fonte:", spark.table(fonte).count())
print("Destino:", spark.table(destino).count())
print("IDs distintos:", spark.table(destino).select("id_evento").distinct().count())

**Resultado esperado:** fonte = **30**, destino = **30**, IDs distintos = **30**. O checkpoint permite consumir apenas os **10 eventos novos**, sem reler os 20 que já foram incorporados pelo streaming.


## Desafio final — Ingestão de arquivos com Auto Loader

**Cenário:** A plataforma de hospedagem passou a disponibilizar seus eventos de navegação em arquivos JSON. Você deverá implementar uma nova forma de ingestão em streaming utilizando o **Auto Loader** do Databricks.

Diferentemente dos ciclos anteriores, em que consumimos uma tabela Delta, agora o streaming deverá identificar e processar arquivos armazenados em uma pasta.

### Atividade

1. Selecione **10 eventos** da tabela `amostra` e grave-os em formato JSON em um Volume do Unity Catalog.
2. Utilize `spark.readStream` com o formato `cloudFiles` para consumir esses arquivos.
3. Defina explicitamente o schema utilizando a estrutura da tabela `amostra`.
4. Grave os eventos em uma nova tabela Delta chamada `aula22_autoloader`, utilizando `writeStream`, `availableNow=True` e um checkpoint próprio.
5. Consulte a tabela de destino e confirme que os 10 eventos foram ingeridos.

### Questões para análise

- Qual é a diferença entre `spark.readStream.table()` e `spark.readStream.format("cloudFiles")`?
- Como o Auto Loader identifica os arquivos que ainda não foram processados?
- Qual é a função do checkpoint nessa nova forma de ingestão?

**Resultado esperado:** uma nova tabela Delta contendo os 10 eventos provenientes dos arquivos JSON.

In [ ]:

# DESAFIO FINAL — INGESTÃO COM AUTO LOADER

# 1. Preparação dos caminhos
pasta_json = f"/Volumes/{catalogo}/{esquema}/{volume}/desafio_json"
checkpoint_json = f"/Volumes/{catalogo}/{esquema}/{volume}/checkpoint_autoloader"
destino_json = f"`{catalogo}`.`{esquema}`.`aula22_autoloader`"

# Reinicia o desafio, permitindo executá-lo novamente
dbutils.fs.rm(checkpoint_json, True)
spark.sql(f"DROP TABLE IF EXISTS {destino_json}")

# 2. Seleção e gravação de 10 eventos em JSON
(spark.table(amostra)
    .filter(F.col("id_evento") <= 10)
    .write
    .mode("overwrite")
    .json(pasta_json))

# 3. Ingestão com Auto Loader
df_json = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .schema(spark.table(amostra).schema)
    .load(pasta_json))

# 4. Gravação incremental na tabela Delta
consulta = (df_json.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_json)
    .trigger(availableNow=True)
    .toTable(destino_json))

consulta.awaitTermination()

# 5. Verificação dos resultados
print("Eventos ingeridos:", spark.table(destino_json).count())
display(spark.table(destino_json).orderBy("id_evento"))

### Orientações para a discussão final

- **Relatório mensal:** **batch**, pois o fechamento é periódico e não exige atualização a cada evento.
- **Indicador operacional:** **streaming**, idealmente contínuo se o requisito de latência for baixo; a demonstração em `availableNow=True` é um exemplo de ingestão incremental por execução.
- **Painel misto:** **arquitetura híbrida**, reunindo agregados históricos e atualizações de novos eventos.

**Questão central:** a segunda execução do streaming não processou novamente os 20 registros anteriores. Com os 10 registros novos, o resultado final permaneceu em 30, e não 50.

**Atenção ao reuso:** se o aluno executar uma célula de inserção duas vezes sem reiniciar a preparação, poderá repetir eventos. Checkpoint evita reprocessar versões da fonte, mas não equivale a deduplicação por `id_evento`.

---
**Referências oficiais:**
- [Wanderbricks — dataset do catálogo samples](https://docs.databricks.com/aws/en/discover/wanderbricks-dataset)
- [Structured Streaming lendo e escrevendo em Delta Lake](https://docs.databricks.com/aws/en/structured-streaming/delta-lake)
- [Streaming em compute serverless: availableNow](https://docs.databricks.com/aws/en/compute/serverless/streaming)